<div style="background:#A6192E;color:#fff;padding:22px 28px;border-radius:14px;font-family:'Barlow Condensed','Arial Narrow',Arial,sans-serif">
<div style="letter-spacing:.2em;text-transform:uppercase;font-size:13px;opacity:.9">Python para Finanzas y Análisis de Datos · Sesión 2</div>
<div style="font-size:34px;font-weight:300;text-transform:uppercase;margin-top:6px"><span style="background:#fff;color:#A6192E;border-radius:10px;padding:2px 12px;margin-right:14px;font-weight:500">2.4</span>Valoración con incertidumbre</div>
</div>

**Objetivo.** pasar de un DCF con una cifra a un DCF que entrega una probabilidad.

**Lo que vamos a hacer**

- Programar un DCF vectorizado.
- Dar distribuciones a los supuestos clave.
- Obtener la distribución del valor por acción y P(valor > precio).

In [ ]:
import sys, pathlib, subprocess, warnings
warnings.filterwarnings("ignore")

# Localiza la carpeta del curso (en Google Colab clona el repositorio)
root = pathlib.Path.cwd().resolve()
while root != root.parent and not (root / "common").exists():
    root = root.parent
if not (root / "common").exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/Ulysess824/Python-for-Finance"], check=False)
    root = pathlib.Path("Python-for-Finance/python-finance-course").resolve()
sys.path.insert(0, str(root))
# En Colab, ejecuta antes:  !pip install -q yfinance arch

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from common import *
set_plot_style()

## 1. Supuestos base

In [ ]:
fcf0 = 100.0       # flujo de caja libre actual (millones)
net_debt = 300.0   # deuda neta (millones)
shares = 50.0      # acciones (millones)
H = 5              # años proyectados
price = 30.0       # precio de mercado por acción

> ### ✏️ Ejercicio 1 — dcf_value
> Valor por acción: FCF crece a `g` por H años, se descuenta a `wacc` y se añade valor terminal de Gordon con `gt`. Debe aceptar **arreglos** (para simular). Con g=8%, wacc=9%, gt=2% da ≈ 31,6.

In [ ]:
def dcf_value(g, wacc, gt):
    g, wacc, gt = (np.atleast_1d(x).astype(float) for x in (g, wacc, gt))
    # TODO
    return None

print(dcf_value(0.08, 0.09, 0.02))

In [ ]:
def dcf_value(g, wacc, gt):
    g, wacc, gt = (np.atleast_1d(x).astype(float) for x in (g, wacc, gt))
    t = np.arange(1, H + 1)
    fcf = fcf0 * (1 + g[:, None]) ** t
    pv = (fcf / (1 + wacc[:, None]) ** t).sum(axis=1)
    tv = fcf[:, -1] * (1 + gt) / (wacc - gt) / (1 + wacc) ** H
    return (pv + tv - net_debt) / shares

print(dcf_value(0.08, 0.09, 0.02))

## 2. Monte Carlo sobre los supuestos

Nadie conoce el crecimiento ni el WACC exactos. Les damos distribuciones: $g\sim N(8\%,3\%)$, WACC $\sim N(9\%,1\%)$, $g_{term}\sim N(2\%,0{,}5\%)$.

> ### ✏️ Ejercicio 2 — Simular el valor
> Sortea 100.000 combinaciones, descarta las absurdas (WACC ≤ g_term + 1%) y calcula `vals`.

In [ ]:
N = 100_000
rng = np.random.default_rng(1)
# TODO: vals = ...
vals = None

In [ ]:
N = 100_000
rng = np.random.default_rng(1)
g = rng.normal(0.08, 0.03, N)
wacc = rng.normal(0.09, 0.01, N)
gt = rng.normal(0.02, 0.005, N)
ok = wacc > gt + 0.01
vals = dcf_value(g[ok], wacc[ok], gt[ok])
print(f"{ok.mean():.1%} de los escenarios son válidos")

## 3. Resultado como probabilidad

> ### ✏️ Ejercicio 3 — P(valor > precio)
> Calcula P5/P50/P95 y la probabilidad de que el valor intrínseco supere el precio de mercado. Grafica el histograma con una línea en el precio.

In [ ]:
# TODO

In [ ]:
p5, p50, p95 = np.percentile(vals, [5, 50, 95])
prob = (vals > price).mean()
print(f"P5 {p5:.1f} | P50 {p50:.1f} | P95 {p95:.1f}  (precio mercado {price})")
print(f"Probabilidad de que valga más que el precio: {prob:.1%}")

fig, ax = plt.subplots()
ax.hist(vals[(vals > -20) & (vals < 80)], bins=100, color="#d9d9d9")
ax.axvline(price, color="#A6192E", ls="--", label=f"Precio = {price}")
ax.axvline(p50, color="#1f1f1f", label=f"Mediana = {p50:.1f}")
ax.set_title("Valor por acción (DCF Monte Carlo)"); ax.legend(); plt.show()

## Para llevarse

- Un DCF puntual da una falsa precisión; una distribución muestra cuánto depende la valoración de los supuestos.
- «Hay 70% de probabilidad de que valga más que el precio» es más honesto que «vale 14».
- El valor terminal domina el resultado: revisa siempre WACC − g_term.

---
*Excel para mirar, Python para repetir.*